# 01.1 — Unicode, Encoding & Text Normalization

**Problem it solves:** Before any NLP, text is bytes. Understanding how bytes become characters — and where that goes wrong — prevents a whole class of silent bugs.

**Why it was invented:** Computers only store numbers. ASCII handled English with 128 characters. The world has ~150,000+ characters across 100+ scripts. Unicode is the universal standard that assigns a unique number (code point) to every character. Encodings (UTF-8, UTF-16) define how to store those numbers as bytes.

---

## Part 1: Code Points vs. Bytes

In [ ]:
# A Unicode code point is just an integer
char = 'A'
print(f"Character: {char}")
print(f"Code point (int): {ord(char)}")
print(f"Code point (hex): U+{ord(char):04X}")
print(f"Back to char: {chr(65)}")

# Now something non-ASCII
for c in ['é', '中', '🔥', 'ñ']:
    print(f"{c!r}  U+{ord(c):04X}  UTF-8 bytes: {c.encode('utf-8').hex()}")

In [ ]:
# UTF-8 is variable-width: ASCII chars use 1 byte, others use 2-4
for c in ['A', 'é', '中', '🔥']:
    encoded = c.encode('utf-8')
    print(f"{c!r}  len(str)={len(c)}  len(bytes)={len(encoded)}  bytes={list(encoded)}")

**Key insight:** `len(string)` counts code points, NOT bytes. An emoji is 1 character but 4 bytes. This matters when you talk to APIs that charge by bytes or truncate at byte limits.

## Part 2: The Normalization Problem

The famous `café != café` bug.

In [ ]:
import unicodedata

# 'é' can be represented TWO ways in Unicode:
# NFC: single precomposed character U+00E9
# NFD: base character 'e' + combining accent U+0301

s1 = 'café'          # NFC form (precomposed)
s2 = 'cafe\u0301'    # NFD form (e + combining accent)

print(f"s1: {s1!r}  len={len(s1)}  bytes={s1.encode('utf-8').hex()}")
print(f"s2: {s2!r}  len={len(s2)}  bytes={s2.encode('utf-8').hex()}")
print(f"s1 == s2: {s1 == s2}")   # False! Same visual, different bytes
print(f"They look the same: '{s1}' vs '{s2}'")

In [ ]:
# Fix with Unicode normalization
s1_nfc = unicodedata.normalize('NFC', s1)
s2_nfc = unicodedata.normalize('NFC', s2)
print(f"After NFC normalization: {s1_nfc == s2_nfc}")

# NFC vs NFD vs NFKC vs NFKD
# NFC:  canonical composition   (preferred for storage)
# NFD:  canonical decomposition
# NFKC: compatibility composition (collapses visually similar chars — good for NLP)
# NFKD: compatibility decomposition

# NFKC example: fullwidth vs regular ASCII
fullwidth = 'Ａ Ｂ Ｃ'  # These are "fullwidth" Latin chars common in East Asian text
print(f"Fullwidth: {fullwidth!r}")
print(f"After NFKC: {unicodedata.normalize('NFKC', fullwidth)!r}")

## Part 3: Build a Text Normalizer from Scratch

In [ ]:
import unicodedata
import re

def normalize_text(text: str, 
                   unicode_form: str = 'NFKC',
                   lowercase: bool = True,
                   strip_accents: bool = False) -> str:
    """
    Normalize text for NLP tasks.
    
    - unicode_form: NFKC collapses compatibility chars (fullwidth, ligatures)
    - strip_accents: remove combining diacritical marks (good for romanization)
    """
    # Step 1: Unicode normalization
    text = unicodedata.normalize(unicode_form, text)
    
    # Step 2: Optionally strip accent marks (combining characters)
    if strip_accents:
        # Decompose first (NFD splits base char + combining mark)
        text = unicodedata.normalize('NFD', text)
        # Keep only non-combining characters
        text = ''.join(c for c in text 
                       if unicodedata.category(c) != 'Mn')
    
    # Step 3: Lowercase
    if lowercase:
        text = text.lower()
    
    # Step 4: Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test it
test_cases = [
    'cafe\u0301',           # NFD form of 'café'
    'Ａ　ＦＵＬＬＷＩＤＴＨ　ｓｅｎｔｅｎｃｅ',  # fullwidth
    '  too   many    spaces  ',
    'UPPERCASE TEXT',
    'Héllo Wörld',
]

for t in test_cases:
    print(f"Input:  {t!r}")
    print(f"Output: {normalize_text(t)!r}")
    print(f"Strip accents: {normalize_text(t, strip_accents=True)!r}")
    print()

## Part 4: Encoding Errors in the Wild

In [ ]:
# Common real-world encoding bugs

# 1. Mojibake: UTF-8 bytes read as Latin-1
original = 'naïve café'
mojibake = original.encode('utf-8').decode('latin-1')  # wrong codec
print(f"Original:  {original}")
print(f"Mojibake:  {mojibake}")
print(f"Fixed:     {mojibake.encode('latin-1').decode('utf-8')}")
print()

# 2. Handling unknown bytes gracefully
messy_bytes = b'Hello \xff World'  # \xff is invalid in UTF-8
print("Strict:", end=' ')
try:
    print(messy_bytes.decode('utf-8', errors='strict'))
except UnicodeDecodeError as e:
    print(f"Error: {e}")
    
print(f"Ignore:  {messy_bytes.decode('utf-8', errors='ignore')!r}")
print(f"Replace: {messy_bytes.decode('utf-8', errors='replace')!r}")

In [ ]:
# 3. Unicode categories — useful for filtering
# Every Unicode character has a category like Lu (uppercase letter), 
# Nd (decimal digit), Ps (open punctuation), etc.

sample = 'Hello, World! 123 🌍 ñ'
for char in sample:
    cat = unicodedata.category(char)
    name = unicodedata.name(char, 'UNKNOWN')
    print(f"  {char!r:6} category={cat:3} ({name})")

In [ ]:
# Build a character-category-based filter
def keep_only(text: str, categories: set) -> str:
    return ''.join(c for c in text if unicodedata.category(c) in categories)

text = 'Hello, World! 123 🌍 café'

# Keep only letters and spaces
letters = {'Lu', 'Ll', 'Lt', 'Lm', 'Lo', 'Zs'}  # L* = letters, Zs = space
print(f"Letters only: {keep_only(text, letters)!r}")

# Keep letters + digits
alphanumeric = letters | {'Nd'}  # Nd = decimal digit
print(f"Alphanumeric: {keep_only(text, alphanumeric)!r}")

## Part 5: What Breaks?

Try to break the normalizer. Run these and explain what happens.

In [ ]:
# Edge case 1: Emoji with skin tone modifiers
# 👋 + skin tone modifier = 👋🏽 (but they're two code points!)
wave = '👋'
wave_brown = '👋🏽'
print(f"wave len={len(wave)}, wave_brown len={len(wave_brown)}")
print(f"bytes: {len(wave.encode('utf-8'))} vs {len(wave_brown.encode('utf-8'))}")

# Edge case 2: Turkish dotless-i — lowercase('I') != 'i' in Turkish locale
# Python's str.lower() is locale-unaware
print(f"'I'.lower() = {'I'.lower()!r}")
# In Turkish: uppercase 'I' -> lowercase 'ı' (dotless i)
# This is why case-folding Turkish text is non-trivial

# Edge case 3: Zero-width characters (often used to hide text)
zwj = '\u200d'  # Zero-width joiner
text_with_zwj = f'hel{zwj}lo'
print(f"With ZWJ: {text_with_zwj!r}  looks like: '{text_with_zwj}'  len={len(text_with_zwj)}")
print(f"Are they equal? {'hello' == text_with_zwj}")

## Summary

**What it does:** Maps human-readable text to a canonical byte representation your code can reliably process.

**When to use it:** At every system boundary — when reading files, accepting user input, calling APIs, or comparing strings.

**What breaks it:**
- Multi-codepoint characters (emoji with modifiers, flags) break naive `len()` assumptions
- Locale-sensitive operations (Turkish i, German ß) are not handled by Python's built-ins
- Zero-width characters can make strings look identical but compare unequal
- NFKC can be too aggressive — it collapses ² (superscript) to 2, which changes mathematical meaning

---

**Next:** 01.2 — Regex for NLP: extracting structure from raw text